In [ ]:
import os
import sys
import boto3
import pandas as pd

# Allow importing from apps/api
sys.path.insert(0, os.path.abspath("../../../../apps/api"))

from livewell.ingestion.s3 import read_parquet, write_parquet
from livewell.ingestion.constants import INSTRUMENTS, INTERVALS

BUCKET = os.environ["LIVEWELL_BUCKET"]
SIGNALS_PREFIX = "signals"
PRICES_PREFIX = "prices"
OUTPUT_PATH = "../../../data/phase2/labeled_signals.parquet"
INTERVAL = "1d"  # label construction uses daily bars only

In [ ]:
def load_all_parquets(bucket: str, prefix: str) -> pd.DataFrame:
    """List all Parquet objects under prefix and concat into one DataFrame."""
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/")
    objects = resp.get("Contents", [])
    frames = []
    for obj in objects:
        df = read_parquet(bucket, obj["Key"])
        if df is not None:
            frames.append(df)
    if not frames:
        raise ValueError(f"No Parquet files found under s3://{bucket}/{prefix}/")
    return pd.concat(frames, ignore_index=True)


def load_signals(bucket: str, s3_key: str, interval: str) -> pd.DataFrame:
    """Load all signal Parquets for one instrument+interval, add s3_key column."""
    prefix = f"{SIGNALS_PREFIX}/{s3_key}/{interval}"
    df = load_all_parquets(bucket, prefix)
    df["s3_key"] = s3_key
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df


def load_prices(bucket: str, s3_key: str, interval: str) -> pd.DataFrame:
    """Load all price Parquets for one instrument+interval, return date+close only."""
    prefix = f"{PRICES_PREFIX}/{s3_key}/{interval}"
    df = load_all_parquets(bucket, prefix)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df[["date", "close"]].rename(columns={"close": "close_d"})


def derive_label(signals: pd.DataFrame, prices: pd.DataFrame) -> pd.DataFrame:
    """
    For each signal on day D, find the D+1 close (close_d1) and derive
    binary ITM label:
      - direction=="buy":  label=1 if close_d1 >= strike_candidate else 0
      - direction=="sell": label=1 if close_d1 <= strike_candidate else 0
      - direction=="none": label=NaN (excluded from training)
    """
    prices_sorted = prices.sort_values("date").reset_index(drop=True)
    prices_sorted["date_d"] = prices_sorted["date"]
    prices_sorted["close_d1"] = prices_sorted["close_d"].shift(-1)

    merged = signals.merge(
        prices_sorted[["date_d", "close_d1"]].rename(columns={"date_d": "date"}),
        on="date",
        how="left",
    )

    def _label(row):
        if row["direction"] == "buy":
            return int(row["close_d1"] >= row["strike_candidate"])
        elif row["direction"] == "sell":
            return int(row["close_d1"] <= row["strike_candidate"])
        else:
            return float("nan")

    merged["label"] = merged.apply(_label, axis=1)
    return merged


def build_labeled_dataset(bucket: str, interval: str) -> pd.DataFrame:
    """
    Load signals and prices for all instruments, derive labels, return combined DataFrame.
    Rows with direction=="none" are included but label=NaN.
    """
    all_frames = []
    for inst in INSTRUMENTS:
        s3_key = inst["s3_key"]
        try:
            signals = load_signals(bucket, s3_key, interval)
            prices = load_prices(bucket, s3_key, interval)
            labeled = derive_label(signals, prices)
            all_frames.append(labeled)
            print(f"{s3_key}: {len(labeled)} rows, {labeled['label'].notna().sum()} labeled")
        except Exception as exc:
            print(f"WARNING: {s3_key} failed — {exc}")
    if not all_frames:
        raise RuntimeError("No instruments produced labeled data.")
    return pd.concat(all_frames, ignore_index=True)

In [ ]:
labeled = build_labeled_dataset(BUCKET, INTERVAL)
print(f"\nTotal rows: {len(labeled)}")
print(f"Labeled (direction != none): {labeled['label'].notna().sum()}")
print(f"Label distribution:\n{labeled['label'].value_counts(dropna=True)}")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
labeled.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

In [ ]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[["date", "s3_key", "direction", "strike_candidate", "close_d1", "label", "signal_valid"]].head(20).to_string())